<a href="https://colab.research.google.com/github/doannguyenminhtamcuc-ship-it/Data_VN_chapter1/blob/main/T%E1%BB%95ng%20h%E1%BB%A3p%20code%20ph%C3%A2n%20t%C3%ADch%20Ch%C6%B0%C6%A1ng%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Xử lý dữ liệu thiếu và tạo biến

Hình 15: Hiển thị dữ liệu bằng thư viện Pandas trong Google Colab

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Employment OECD INDUSTRY.csv')
display(df.head())

Hình 16: Kiểm tra và xử lý dữ liệu thiếu bằng phương pháp nội suy tuyến tính trong Pandas

In [ ]:
# Check for missing values
print("Missing values before filling:\n", df.isnull().sum())

# Fill missing values with the average of the previous and next year
for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].interpolate(method='linear', limit_direction='both'))

print("\nMissing values after filling:\n", df.isnull().sum())

display(df.head())

Hình 17: Tạo biến mới 'Tỷ lệ lao động nữ so với lao động nam’  (female-male ratio) sau khi định hình lại cấu trúc dữ liệu

In [ ]:
# Pivot the data to have 'Female' and 'Male' employment as separate columns
pivot_df = df.pivot_table(
    index=['Country', 'Year', 'Economic activity'],
    columns='Gender',
    values='Employment Thousands'
).reset_index()

# Create the 'Female-Male Ratio' variable
pivot_df['Female-Male Ratio'] = pivot_df['Female'] / pivot_df['Male']

# Display the first few rows of the new dataframe with the calculated ratio
display(pivot_df.head())

# Phân tích việc làm theo giới tính: trực quan hóa và diễn giải kinh tế

Hình 18: Vẽ biểu đồ đường bằng Altair thể hiện xu hướng 'Tỷ lệ lao động nữ so với lao động nam' qua các năm cho từng lĩnh vực kinh tế

In [ ]:
import altair as alt

# Create the line chart
chart = alt.Chart(pivot_df).mark_line().encode(
    x=alt.X('Year:O', title='Year'),  # Use ordinal for year
    y=alt.Y('Female-Male Ratio', title='Female-Male Ratio'),
    color='Economic activity:N',  # Color lines by economic activity
    tooltip=['Year', 'Economic activity', 'Female-Male Ratio']
).properties(
    title='Female-Male Ratio Over Years by Economic Activity'
).interactive() # Make the chart interactive for zooming and panning

# Display the chart
chart.display()

Hình 20: Lệnh Altair vẽ biểu đồ thanh thể hiện 'Tỷ lệ lao động nữ so với lao động nam theo ngành kinh tế trong năm 2024, có sắp xếp theo thứ tự và thêm nhãn giá trị

In [ ]:
import altair as alt

# Filter data for the year 2024
df_2024 = pivot_df[pivot_df['Year'] == 2024]

# Sort the dataframe by 'Female-Male Ratio' in descending order
df_2024 = df_2024.sort_values('Female-Male Ratio', ascending=False)

# Create the bar chart
chart = alt.Chart(df_2024).mark_bar().encode(
    x=alt.X('Economic activity Short', title='Economic Activity', sort='-y', axis=alt.Axis(labelAngle=-45)), # Sort bars by y-axis in descending order and angle labels
    y=alt.Y('Female-Male Ratio', title='Female-Male Ratio'),
    tooltip=['Economic activity Short', 'Female-Male Ratio'],
    color=alt.Color('Economic activity Short', legend=None) # Add color based on economic activity, remove legend
).properties(
    title='Female-Male Ratio by Economic Activity in 2024',
    width=800,  # Set the width of the chart
    height=500  # Set the height of the chart
)

# Add text labels on top of the bars
text = chart.mark_text(
    align='center',
    baseline='bottom',
    dy=-5  # Nudge text up a bit
).encode(
    text=alt.Text('Female-Male Ratio', format='.2f'), # Format the ratio to 2 decimal places
    color=alt.value('black')  # Set the color of the text
)

# Combine the bar chart and the text
final_chart = chart + text

# Display the chart
final_chart.display()

# Phân tích khoảng cách thu nhập giới tính: trực quan hóa và diễn giải kinh tế

In [ ]:
import pandas as pd

# Load the Earnings CSV file
earnings_df = pd.read_csv("/content/Earnings lọc mới nhất.csv", encoding='latin1')

print("Earnings DataFrame head:")
display(earnings_df.head())

Hình 22: Lệnh và kết quả tạo biến khoảng cách thu nhập giữa nam và nữ

In [ ]:
print(earnings_df.columns)
earnings_pivoted = earnings_df.pivot_table(index=['Country ', 'Year', 'Economic activity'], columns='Gender', values='Average monthly earnings  $PPP 2021')
earnings_pivoted['Gender Earning Gap'] = earnings_pivoted['Male'] - earnings_pivoted['Female']
earnings_pivoted = earnings_pivoted.fillna(0)
display(earnings_pivoted.head())

Hình 23: Lệnh và kết quả gộp dữ liệu biến khoảng cách thu nhập các quốc gia OECD năm 2024

In [ ]:
# Filter for the year 2024
earnings_2024 = earnings_pivoted.loc[(slice(None), 2024, slice(None)), :]

# Group by Economic activity and calculate the mean of Gender Earning Gap
average_gender_earning_gap_2024 = earnings_2024.groupby('Economic activity')['Gender Earning Gap'].mean()

print("Average Gender Earning Gap by Economic Activity for 2024:")
display(average_gender_earning_gap_2024)

Hình 24: Lệnh Altair vẽ biểu đồ thanh thể hiện 'Chênh lệch thu nhập giới tính' theo ngành kinh tế trong năm 2024, có sắp xếp theo thứ tự

In [ ]:
import altair as alt

# Convert the Series to a DataFrame for Altair
average_gender_earning_gap_2024_df = average_gender_earning_gap_2024.reset_index()
average_gender_earning_gap_2024_df.columns = ['Economic activity', 'Average Gender Earning Gap']

# Create the bar chart
chart = alt.Chart(average_gender_earning_gap_2024_df).mark_bar().encode(
    x=alt.X('Economic activity', sort='-y', title='Economic Activity'), # Sort in descending order by y-axis value and add title
    y=alt.Y('Average Gender Earning Gap', title='Average Gender Earning Gap ($PPP 2021)'), # Add y-axis title
    tooltip=['Economic activity', alt.Tooltip('Average Gender Earning Gap', format = ',.2f')], # Add tooltip with formatting
    color=alt.Color('Economic activity', legend=None) # Color by economic activity
).properties(
    title='Average Gender Earning Gap by Economic Activity (OECD, 2024)'
).interactive() # Make the chart interactive for zooming and panning

# Display the chart
chart.display()

# Phân tích suy thoái kinh tế Mexico

In [ ]:
import pandas as pd

file_path = '/content/Mexico 2005-2024.xlsx'
df = pd.read_excel(file_path)
display(df.head())

Hình 26: Lệnh tạo biểu đồ ghép thể hiện 'Các chỉ số vĩ mô, xu hướng thu nhập tỷ lệ lao động nữ so với lao động nam tại Mexico trong các giai đoạn suy thoái và tăng trưởng' giai đoạn 2005 – 2024

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure the DataFrame is sorted by 'YEAR' for chronological plotting
df_sorted = df.sort_values(by='YEAR', ascending=True).copy()

# Create a figure with two vertically stacked subplots, sharing the x-axis
fig, (ax0, ax1) = plt.subplots(2, 1, sharex=True, figsize=(14, 12))

# --- Top Subplot: Macroeconomic Indicators (Line Chart) ---
# Plot Inflation Rate, GDP Growth Rate, and Unemployment Rate (%)
sns.lineplot(x='YEAR', y='Inflation Rate', data=df_sorted, label='Inflation Rate', marker='o', ax=ax0, color='red')
sns.lineplot(x='YEAR', y='GDP Growth Rate', data=df_sorted, label='GDP Growth Rate', marker='o', ax=ax0, color='green')
sns.lineplot(x='YEAR', y='Unemployment rate (%)', data=df_sorted, label='Unemployment Rate (%)', marker='o', ax=ax0, color='purple')

ax0.set_ylabel('Rate (%)', fontsize=12, color='black')
ax0.set_title('Inflation Rate, GDP Growth Rate, and Unemployment Rate 2005-2024 in Mexico', fontsize=14)
ax0.grid(True, linestyle='--', alpha=0.7)
ax0.legend(fontsize=10)

# --- Bottom Subplot: Gender Earnings Trends (Dual-Axis Chart) ---
# Female-to-Male Ratio as bars (left axis)
color_bars = 'lightgreen' # As per previous interaction
ax1.bar(df_sorted['YEAR'], df_sorted['Female-to-Male Ratio'], color=color_bars, label='Female-to-Male Ratio', width=0.6)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Female-to-Male Ratio', color='black', fontsize=12)
ax1.tick_params(axis='y', labelcolor='black') # Y-axis ticks color for ax1
ax1.grid(True, linestyle='--', alpha=0.7, axis='y')

# Create a second y-axis sharing the same x-axis
ax1_twin = ax1.twinx()

# Average Monthly Earnings as a line (right axis)
color_line = 'blue' # As per previous interaction
ax1_twin.plot(df_sorted['YEAR'], df_sorted['Average monthly earnings (2021 PPP $)'], color=color_line, label='Average Monthly Earnings (2021 PPP $)', marker='o', linestyle='-')
ax1_twin.set_ylabel('Average Monthly Earnings (2021 PPP $)', color=color_line, fontsize=12)
ax1_twin.tick_params(axis='y', labelcolor=color_line) # Y-axis ticks color for ax1_twin

ax1.set_title('Female-to-Male Ratio and Average Monthly Earnings 2005-2024 in Mexico', fontsize=14)

# Combine legends for the dual-axis subplot
handles_ax1, labels_ax1 = ax1.get_legend_handles_labels()
handles_ax1_twin, labels_ax1_twin = ax1_twin.get_legend_handles_labels()
ax1_twin.legend(handles_ax1 + handles_ax1_twin, labels_ax1 + labels_ax1_twin, loc='upper left', fontsize=10)

# Set x-axis ticks for both subplots (since they share x)
ax1.set_xticks(df_sorted['YEAR'])
ax1.set_xticklabels(df_sorted['YEAR'], rotation=45, ha='right') # Rotate x-axis labels for better readability

# --- Add shared light shading for highlighted periods ---
shading_alpha = 0.15 # Transparency for shading
shading_color = 'gray' # Color for shading

# Shading for 2008-2010 (covering years from 2007.5 to 2010.5 for full year highlights)
ax0.axvspan(2007.5, 2010.5, color=shading_color, alpha=shading_alpha)
ax1.axvspan(2007.5, 2010.5, color=shading_color, alpha=shading_alpha)
ax0.text(2009, ax0.get_ylim()[1]*0.95, 'Recession', color='black', ha='center', va='top', fontsize=10)

# Shading for 2019-2021 (covering years from 2018.5 to 2021.5)
ax0.axvspan(2018.5, 2021.5, color=shading_color, alpha=shading_alpha)
ax1.axvspan(2018.5, 2021.5, color=shading_color, alpha=shading_alpha)
ax0.text(2020, ax0.get_ylim()[1]*0.95, 'Recession', color='black', ha='center', va='top', fontsize=10)

# --- Full Figure Title ---
fig.suptitle('Economic Indicators, Earnings, and Gender Ratio Trends in Mexico', fontsize=18, y=1.02)

fig.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent overlap and make space for suptitle
plt.show()